# 1.前言

目标：搞懂**预训练语言模型两大能力：【推理预测】 + 【微调更新权重】**，从分词、向量提取、高层 pipeline、底层模型推理，最后做一轮微调

# 2.tokenizer：文本 → id 序列

In [1]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'   # 把 huggingface 下载源切换为国内镜像，避免 Errno101 Network unreachable
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'      # HEAD/etag 检查超时：默认 10s → 60s
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'  # 文件下载超时：默认 10s → 60s
# 注意：环境变量必须放在 import transformers 之前
# 两个 timeout：拉长网络等待时间，弱网下防止直接超时报错

import torch
import transformers
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
text = 'I love machine learning, and you?'
enc = tokenizer(text, return_tensors = 'pt')
print('input_ids:', enc['input_ids'])
print('attention_mask:', enc['attention_mask'])
print('词表大小:', tokenizer.vocab_size)

input_ids: tensor([[ 101, 1045, 2293, 3698, 4083, 1010, 1998, 2017, 1029,  102]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
词表大小: 30522


要点（这节最重要的概念）：
- BERT 不吃字符串，吃整数 id 序列。tokenizer 干四件事：① 分词（WordPiece 子词切分）② 查词表映射成 id ③ 加特殊 token（句首 [CLS]、句尾 [SEP]）④ 截断/填充 + 生成 attention_mask。return_tensors='pt' 让输出直接是 torch 张量。

`AutoTokenizer.from_pretrained("bert-base-uncased")`
- 自动从镜像下载 BERT‑base‑uncased 分词器（vocab 词表、配置），缓存到本地 ~/.cache/huggingface/hub ；第二次运行直接读本地缓存，不再联网。
-  bert‑base‑uncased ：英文小写版本 BERT，全部文本转小写，不区分大小写。

enc = tokenizer(text, return_tensors = 'pt')
- 调用分词器对句子编码：
  - `return_tensors="pt"`：输出 PyTorch tensor；不写默认输出 python list。
- BERT tokenizer 自动做几件事：
  1. 句子开头插入特殊标记 `[CLS]`
  2. 句子末尾插入特殊标记 `[SEP]`
  3. 分词、转 id；遇到不在词表里的词会做 subword 子词切分
  4. 生成 `input_ids`：token 对应的编号
  5. 生成 `attention_mask`：1 代表真实 token，0 代表 padding 填充位置；本句没有 padding，全部是 1。

- id 含义对照
|id	|token|
|-|-|
|101	|[CLS] 分类起始标记|
|1045	|I|
|2293	|love|
|3698	|machine|
|4083	|learning|
|1011	|,|
|1998	|and|
|2017	|you|
|102	|[SEP] 句子结束标记|
- `input_ids.shape = [1,9]`：`[batch_size, seq_len]`，这里 batch=1，一共 9 个 token。
- `attention_mask`：全部为 1，没有 pad。

# 3.decode 还原 + 看子词

In [2]:
ids = enc['input_ids'][0]
print('id -> 词:', [tokenizer.decode([i]) for i in ids])
# WordPiece 子词演示：## 前缀 = 子词拼接
print(tokenizer.tokenize('unbelievably'))

id -> 词: ['[CLS]', 'i', 'love', 'machine', 'learning', ',', 'and', 'you', '?', '[SEP]']
['un', '##bel', '##ie', '##va', '##bly']


要点：
- id → 词反向映射，看到 [CLS]/[SEP] 长什么样；再演示 WordPiece 子词——## 前缀表示"接在上一子词后面"。亲眼看到"一个词可能被拆成多个子词"是理解 BERT 词表的关键
- `ids = enc['input_ids'][0]`
  - enc['input_ids']  shape： [batch_size, seq_len]，这里 batch=1；
  - `[0]`取出第一条样本，得到一维张量，就是句子对应的 token id 序列
- `[tokenizer.decode([i]) for i in ids]`
> 关键点：`decode([i])`，**传入的是列表`[i]`，不是单个 tensor 数字**。
  - 循环每一个 id，单独解码一个 token；
  - 输出会保留特殊符号`[CLS]`、`[SEP]`

id -> 词: ['[CLS]', 'i', 'love', 'machine', 'learning', ',', 'and', 'you', '[SEP]']
> 区分两种 decode：
1. `tokenizer.decode([i])`：单个 id 解码，只输出对应 token；
2. `tokenizer.decode(ids)`：整条序列解码，自动拼接，输出完整句子：
  - `[CLS] i love machine learning, and you? [SEP]`

`tokenizer.tokenize('unbelievably')`
- BERT 使用 **WordPiece 子词分词**。
  - 完整单词不在词表中，就拆分为多个子词；
  - 子词带前缀`##`，代表**接在前一个 token 后面，拼接时不要加空格**。

['un', '##bel', '##iev', '##ably']
- 拼接：`un + bel + iev + ably = unbelievably`
- WordPiece 规则
  1. 不带`##`：代表新单词的起始子词
  2. 带`##`：接续子词，**和前面的 token 直接拼接，不插入空格**
> `tokenizer.tokenize()`只做分词，**不会自动添加`[CLS]`、`[SEP]`**。
> 只有调用 `tokenizer(text, return_tensors="pt")` 才会自动加上这两个特殊 token
> `decode()`内部会自动消除`##`符号，把子词合并成原始单词。

# 4.AutoModel：前向看形状

In [3]:
from transformers import AutoModel
model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()   # 推理模式：关闭 Dropout、BN；不计算梯度，配合下面 torch.no_grad()
print('参数量：{:.1f}M'.format(sum(p.numel() for p in model.parameters()) / 1e6))
with torch.no_grad():
    out = model(**enc)
print('last_hidden_state:', out.last_hidden_state.shape)
print('pooler_output:', out.pooler_output.shape)
cls_vec = out.last_hidden_state[:, 0, :]
print('CLS 向量形状:', cls_vec.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


参数量：109.5M
last_hidden_state: torch.Size([1, 10, 768])
pooler_output: torch.Size([1, 768])
CLS 向量形状: torch.Size([1, 768])


要点：
- AutoModel 返回 raw hidden states（没有任务头，适合自己接下游）；
- `AutoModel`：加载**BERT 基础模型（不带分类头）**，只输出 Transformer 隐层特征
  >`AutoModelForSequenceClassification` 才是带分类头，用来做文本分类
- 分类任务用 AutoModelForSequenceClassification。last_hidden_state 形状 [batch, 句长, 768]——每个 token 一个 768 维向量；
- 第 0 位是 [CLS] 的向量，常被当作整句表示。model(**enc) 里的 ** 是把 dict 解包成关键字参数 input_ids 、 attention_mask 解包传入模型

print('参数量：{:.1f}M'.format(sum(p.numel() for p in model.parameters()) / 1e6))
- `p.numel()`：统计单个张量元素总数；
- 遍历所有参数求和，除以 10^6，得到百万参数；
- bert‑base‑uncased 总参数量：**109.5M**

`out.last_hidden_state`：最后一层 Transformer 全部 token 的隐向量
- shape：`[batch_size, seq_len, hidden_dim]`
- bert‑base hidden_dim = **768**
- 本例输出：`torch.Size([1, 9, 768])`
  - 第 0 维：batch；第 1 维：每个 token；第 2 维：768 维特征向量。
  - 第 0 个 token 就是 `[CLS]` 位置。

**`out.pooler_output`**：BERT 的池化输出
- shape：`[batch_size, hidden_dim]` → `torch.Size([1, 768])`
> 内部逻辑：取`[CLS]`向量，过一层 Tanh 全连接层得到 pooler_output；
> **注意：pooler_output ≠ 直接取 CLS 向量**，它经过了额外的线性变换 + tanh

cls_vec = out.last_hidden_state[:, 0, :]
- `[:,0,:]`：所有 batch，取第 0 个 token（`[CLS]`），全部 768 维特征；
- shape：`[1,768]`。
> 对比：
> - `cls_vec`：原始`[CLS]`token 输出，**没有经过额外 Tanh 层**；
> - `out.pooler_output`：`[CLS]`再经过一层 Linear+Tanh 变换后的结果。
> 做句子向量表示，**很多时候直接用`last_hidden_state[:,0,:]`，而不是 pooler_output**。

# 5.pipeline：一行推理

In [4]:
from transformers import pipeline

MODEL_DIR = '/home/wsl2/py-learning-log/21-torch/models/distilbert-sst2'  # 本地模型文件夹路径，里面存放：模型权重、config、分词器文件
clf = pipeline('sentiment-analysis', model=MODEL_DIR, tokenizer=MODEL_DIR)
for s in ['This movie is fantastic!', 'The food was terrible.']:
    r = clf(s)[0]
    print(f'{s!r} -> {r["label"]} ({r["score"]:.3f})')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

'This movie is fantastic!' -> POSITIVE (1.000)
'The food was terrible.' -> NEGATIVE (0.999)


要点：
- pipeline 把 HuggingFace 高层封装流水线，把**模型 + 分词器 + 预处理 + 推理**打包，一行调用完成任务，不用手动写 tokenizer、model、eval、no_grad 全封装了，先一行出结果感受效果；

- `clf(s)`：输入文本，返回列表；每个元素是字典；
- `[0]`：取出第一条结果；
- 返回字典包含两个 key：
  - `label`：标签，`POSITIVE` / `NEGATIVE`
  - `score`：概率置信度，0~1 之间

 `{s!r}`
- `!r`：**repr () 格式化**，输出字符串带引号。
> 对比： 
> - `{s}`：直接输出字符串，不带引号：`This movie is fantastic!`
> - `{s!r}`：调用`repr(s)`，输出带单引号：`'This movie is fantastic!'`

`{r["label"]}`
- `r`是 pipeline 返回的字典：`{'label':'POSITIVE', 'score':0.999}`
- `r["label"]` 取出标签字符串：`POSITIVE` / `NEGATIVE`

`({r["score"]:.3f})`
- `r["score"]`：拿到概率浮点数，例如 `0.998723`
- `:.3f`：**浮点数格式化，保留 3 位小数**
  - `0.998723` → `0.999`
  - `0.8121` → `0.812`

# 6.拆开黑盒：手动推理五步

In [5]:
from transformers import AutoModelForSequenceClassification
import torch.nn.functional as F
MODEL_DIR = '/home/wsl2/py-learning-log/21-torch/models/distilbert-sst2'
model2 = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model2.eval()
s = 'This movie is fantastic!'
inputs = tokenizer(s, return_tensors='pt')   # 用 Cell 2 的 tokenizer（bert 的，词表与 distilbert 相同）
with torch.no_grad():
    logits = model2(**inputs).logits
probs = F.softmax(logits, dim=-1)
print('logits:', logits)
print('probs :', probs)
print('预测  :', 'POSITIVE' if probs[0, 1] > 0.5 else 'NEGATIVE')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

logits: tensor([[-4.3476,  4.7046]])
probs : tensor([[1.1712e-04, 9.9988e-01]])
预测  : POSITIVE


要点：
- 恢复是保存的逆操作：新建同结构模型 → load_state_dict 把权重"填"进去。注意 state_dict 不绑定类实例，只要求结构匹配（key 一一对应），这就是它能跨机器移植的原因。torch≥2.0 默认 weights_only=True（防反序列化攻击），存的是纯张量字典所以没影响

`AutoModelForSequenceClassification`：**带分类头的模型**，在 DistilBERT 主干之上额外加一层线性层，直接输出分类 logits，专门做文本分类任务

`F`：torch.nn.functional，提供 softmax 等函数。

logits = model2(**inputs).logits
- `**inputs`解包传入 input_ids、attention_mask；
- `model2(**inputs)`返回对象，`.logits`就是**原始分类得分（未经过 softmax）**；
- shape：`[batch_size, num_classes]` → `torch.Size([1, 2])`。 
> logits 是**原始得分，不是概率**，数值可以是任意实数，可正可负

probs = F.softmax(logits, dim=-1)
- `F.softmax(logits, dim=-1)`：
  - `dim=-1`：在最后一维（类别维度）做 softmax；
  - 将 logits 转换成**概率分布，所有值之和 = 1**；
  - shape 同样`[1,2]`；`probs[0,0]`对应 NEGATIVE 概率，`probs[0,1]`对应 POSITIVE 概率。

# 7.微调雏形：BERT 也是 nn.Module

In [10]:
MODEL_DIR = '/home/wsl2/py-learning-log/21-torch/models/distilbert-sst2'
model3 = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
optimizer = torch.optim.AdamW(model3.parameters(), lr=2e-5)
texts = ['I love this!', 'This is boring.']
batch = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
labels = torch.tensor([1, 0])     # 伪标签
model3.train()
optimizer.zero_grad()
out = model3(**batch, labels=labels)
loss = out.loss
loss.backward()
optimizer.step()
print('微调 loss:', loss.item())
grad_norm = sum(p.grad.norm().item() for p in model3.parameters() if p.grad is not None)
print('梯度范数:', grad_norm, '（非 0 = 梯度流回 BERT 成功）')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

微调 loss: 0.00018505481421016157
梯度范数: 0 （非 0 = 梯度流回 BERT 成功）


要点（预训练-微调范式的最小演示）：
- from_pretrained 返回的对象和 torch 生态完全兼容——.parameters()、.train()/.eval()、接 CrossEntropyLoss 和优化器全都行。
- lr=2e-5 是微调 BERT 的经典小学习率：预训练权重已经很好，学习率大了会灾难性遗忘。
- padding=True 让两句等长（短句补 [PAD]），attention_mask 自动标出哪些是真实词

batch = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
- 一次性编码 2 条句子，构成一个 batch：
1. `padding=True`：两条句子长度不一样时，短句子末尾补`[PAD]`，保证同 shape；自动生成`attention_mask`，padding 位置 mask=0。
2. `truncation=True`：句子超长时自动截断，防止超过模型最大 512 长度限制。
3. 返回字典，包含`input_ids`、`attention_mask`，都是 PyTorch tensor。
> batch 形状：`input_ids: [2, seq_len]`，batch_size=2。

labels = torch.tensor([1, 0])
- SST2 约定：`1=POSITIVE`，`0=NEGATIVE`
- 第 1 句`I love this!` → 标签 1（正面）
- 第 2 句`This is boring.` → 标签 0（负面）

grad_norm = sum(p.grad.norm().item() for p in model3.parameters() if p.grad is not None)
- 遍历模型**所有参数**，筛选出**存在梯度的参数**，分别计算每个参数梯度的 L2 范数，全部加起来求和，得到一个标量。用来**调试：看反向传播有没有成功把梯度传到模型参数上**。

`if p.grad is not None`
- 过滤：**只保留有梯度的参数**
- 如果参数冻结 `requires_grad=False`，`p.grad` 永远是 `None`，直接跳过，不加进求和
- 正常可更新参数，backward 之后`p.grad`是梯度张量

3. `p.grad.norm()`
- `norm()` 默认就是 **L2 范数（二范数）**
- 对于一个梯度张量，L2 范数 = 张量内部所有元素平方求和再开根号。
> 物理含义：衡量这一组梯度整体有多大。梯度向量越长，范数越大，代表本次更新的幅度越大。

# 8.测试小结

1.tokenizer 对输入文本做了哪几件事？return_tensors='pt' 是干什么的？BERT 为什么不能直接吃字符串？
- tokenizer 主要干了干四件事：1.分词（WordPiece 子词切分）；2. 查词表映射成 id；3.加特殊 token（句首 [CLS]、句尾 [SEP]）；4. 截断/填充 + 生成 attention_mask。return_tensors='pt' 让输出直接是 torch 张量。
- return_tensors="pt"：输出 PyTorch 张量;不写则默认输出 python list
- BERT 是神经网络，只能读数字张量，看不懂人类的文字字符串；必须先把文字转成模型词表里对应的数字 ID

2.input_ids 里的 101、102、0 分别是什么 token？attention_mask 全 1 和含 0 各代表什么？Cell 7 里 padding=True 后 mask 会变成什么样？
- 101 是 [CLS] 分类起始标记；102 是 [SEP] 句子结束标记；0 是填充符
- 1 代表真实 token，0 代表 padding 填充位置；
- 两条句子长度不一样时，短句子末尾补[PAD]，保证同 shape，自动生成attention_mask，padding 位置 mask=0。

3.AutoModel 和 AutoModelForSequenceClassification 的区别？last_hidden_state 形状 [1, 8, 768] 的三个维度分别是什么？
- AutoModel：加载BERT 基础模型（不带分类头），只输出 Transformer 隐层特征
- AutoModelForSequenceClassification 才是带分类头，用来做文本分类
- 三个维度分别是 [batch_size, seq_len, hidden_dim]

4.model(**enc) 里的 ** 在干什么？为什么 enc 这个 dict 能被解包成关键字参数传给模型？
- ** 是把 dict 解包成关键字参数 input_ids 、 attention_mask 解包传入模型

5.微调时 lr 为什么用 2e-5 这么小的值？你用什么证据证明"梯度真的流回了 BERT 权重"？（提示：Cell 7 打印了什么？）
- BERT 微调使用 2e-5 小学习率，是为了避免大学习率破坏预训练学到的语言知识、发生灾难性遗忘，仅对权重做小幅修正；依靠代码打印的 grad_norm 作为证据，grad_norm>0 就证明梯度成功回传到 BERT 主干